# 03 · Data-ingestion throughput

Runs only the `ingestion_throughput` suite: Arrow-table-to-Bar conversion (Stage A), single-threaded `MarketDataBus::publish()` fan-out (Stage B), and concurrent-publisher contention (Stage C). Synthetic in-memory data only -- no socket or disk I/O. See docs/superpowers/specs/2026-08-21-tier1-benchmarking-design.md section 3.2.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "python"))
import os; os.chdir(ROOT)
print("repo root:", ROOT)

In [ ]:
from algogauge import manifest, runner, cli
m = manifest.load(ROOT / "algogauge.toml")
suite = m.get("ingestion_throughput")
res = runner.run_suite(suite, m.defaults, ROOT, skip_perf=False)
print(f"run {res.run_id}, perf={'on' if res.perf_ran else 'off'}")

## Results

In [ ]:
from IPython.display import Markdown, display
display(Markdown(cli.summary_markdown(res.records)))

## Stage A: conversion throughput vs. table size

Rows/second implied by median latency, one point per `rows` variant.

In [ ]:
import pandas as pd
import plotly.express as px

conv = pd.DataFrame([
    {"rows": int(r.param), "median_us": r.median, "rows_per_sec": r.counters.get("items_per_second")}
    for r in res.records
    if r.family == "ArrowToBars" and r.param is not None
]).sort_values("rows")
display(conv)
if not conv.empty:
    px.line(conv, x="rows", y="median_us", markers=True, log_x=True, log_y=True,
           title="Arrow-to-Bar conversion latency vs. table size").show()

## Stage C: contention -- latency vs. concurrent publisher threads

Rising median latency as `threads` increases is the mutex contention in `MarketDataBus::publish()` becoming visible.

In [ ]:
cont = pd.DataFrame([
    {"threads": int(r.param), "median_us": r.median, "p95_us": r.p95}
    for r in res.records
    if r.family == "PublishContended" and r.param is not None
]).sort_values("threads")
display(cont)
if not cont.empty:
    px.line(cont.melt(id_vars="threads", value_vars=["median_us", "p95_us"]),
           x="threads", y="value", color="variable", markers=True,
           title="MarketDataBus::publish() contention vs. thread count").show()

## Flamegraph (if perf ran)

In [ ]:
flamegraph = res.result_dir / "flamegraph.svg"
if flamegraph.exists():
    from IPython.display import SVG, display as _display
    _display(SVG(filename=str(flamegraph)))
else:
    print("no flamegraph for this run (perf was skipped or unavailable)")